# Day 10 — Code-acting agents: one `run_python` tool vs many JSON tools
**Theory:** [day-10-code-and-computer-use-agents.md](../theory/day-10-code-and-computer-use-agents.md) · ⏱ ~60 min · Level: medium

⚠️ The local `run_python` here is a **learning sandbox only** (subprocess + timeout). Production needs a
container or a managed sandbox (see the theory note).

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
# Please don't let an AI "auto-fix" rewrite the imports below: the course packages are `course_utils` and `shopease`.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone (or update) the course repo, install it, read the key from notebook secrets
    BRANCH = os.getenv("COURSE_BRANCH", "main")
    REPO_URL = os.getenv("COURSE_REPO_URL", "https://github.com/Rahulverma5/agentic-ai-course")
    REPO = pathlib.Path(os.getenv("COURSE_BASE_DIR", "/content" if IN_COLAB else "/kaggle/working")) / "agentic-ai-course"
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    else:  # an older clone from a previous session: bring it up to date
        subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "reset", "-q", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt"), "-e", str(REPO)],
                   check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
    except Exception as e:
        print("Add GEMINI_API_KEY to the notebook secrets (🔑 icon in Colab, Add-ons → Secrets in Kaggle):", e)
else:  # Codespaces / local: use the checked-out repo
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
try:
    from google.genai import types  # Gemini SDK types: Content, Part, GenerateContentConfig, ...
    from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of, thinking, user
    from shopease.data import setup_data
except ImportError as e:
    raise ImportError(
        f"{e}\n\nThe course code at {REPO} is missing or outdated. Fix it like this:\n"
        f"  • Colab/Kaggle: run  !rm -rf {REPO}  then Runtime → Restart session, and run this cell again.\n"
        "  • Codespaces/local: run  git pull && pip install -e .  in a terminal, then restart the kernel.") from None
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup()                                   # checks your key and picks models your key can use
from course_utils.llm import MODEL, CHEAP_MODEL  # re-read: check_setup() may have switched them

## 1. Make a bigger dataset (500 orders) as CSV

In [ ]:
import random, subprocess, tempfile, json, sqlite3
import pandas as pd
from course_utils import run_agent
rng = random.Random(1)
items = [("Wireless earbuds", 49), ("Running shoes", 89), ("Coffee grinder", 35), ("Laptop stand", 29), ("Smart watch", 199)]
rows = []
for i in range(500):
    item, price = rng.choice(items)
    rows.append({"order_id": f"X{i:04}", "month": f"2026-{rng.randint(1, 10):02}", "item": item, "amount": price,
                 "tier": rng.choice(["gold", "standard", "standard"]), "returned": rng.random() < (0.25 if price > 100 else 0.08)})
DATA = pathlib.Path("week-2/practice/scratch/orders_500.csv"); DATA.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows).to_csv(DATA, index=False)
pd.read_csv(DATA).head()

## 2. The code tool

In [ ]:
def run_python(code: str) -> str:
    """LEARNING ONLY: subprocess + timeout. Not a security boundary."""
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(code)
    try:
        p = subprocess.run([sys.executable, f.name], capture_output=True, text=True, timeout=20, cwd=str(REPO))
        return (p.stdout + p.stderr)[-3000:] or "(no output; use print())"
    except subprocess.TimeoutExpired:
        return "Error: timed out after 20s"

PY_TOOL = {"name": "run_python", "description": f"Run a Python script and get its stdout. pandas is installed. "
           f"The orders dataset is at '{DATA}' (columns: order_id, month, item, amount, tier, returned). Always print() results.",
           "input_schema": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}}
print(run_python("print(2 + 2)"))

## 3. Ask analytics questions

In [ ]:
QUESTIONS = ["What is the return rate per item? Sort highest first.",
             "Which month had the highest revenue from gold customers, and how much?",
             "Is the return rate for items over $100 higher than for cheaper items? Give both numbers."]
code_rows = []
for q in QUESTIONS:
    c0 = tracker.total()
    res = run_agent(q, [PY_TOOL], {"run_python": run_python}, thinking_level="low", verbose=False)
    code_rows.append({"question": q[:50], "steps": res.steps, "cost $": round(tracker.total() - c0, 4), "answer": res.text[:160]})
pd.DataFrame(code_rows)

## 4. Same questions with JSON tools only
A "get_orders(page)" tool returns raw rows. Watch steps and tokens explode, because every row passes through the context.

In [ ]:
ALL = pd.read_csv(DATA)
def get_orders(page: int = 0) -> str:
    return ALL.iloc[page * 100:(page + 1) * 100].to_json(orient="records")
JSON_TOOL = {"name": "get_orders", "description": "Get 100 orders per page (pages 0-4) as JSON rows.",
             "input_schema": {"type": "object", "properties": {"page": {"type": "integer"}}, "required": ["page"]}}
c0, before = tracker.total(), len(tracker.calls)
res = run_agent(QUESTIONS[0], [JSON_TOOL], {"get_orders": get_orders},
                thinking_level="low", verbose=False, max_steps=10)
print(f"JSON tools: steps={res.steps}  input tokens={sum(c['input'] for c in tracker.calls[before:]):,}  "
      f"cost=${tracker.total() - c0:.4f}\n{res.text[:300]}")

## 5. Hosted sandbox: Gemini's built-in code execution tool
Google runs the sandbox, so nothing executes on your machine. The response contains the code it wrote
(`executable_code`) and the result (`code_execution_result`).

In [ ]:
r = client.models.generate_content(
    model=MODEL, contents="Simulate 10,000 rolls of two dice. Report the mean of the sum and the most common sum.",
    config=types.GenerateContentConfig(tools=[types.Tool(code_execution=types.ToolCodeExecution())]))
tracker.add(MODEL, r.usage_metadata, "code_execution tool")
for p in r.candidates[0].content.parts:
    if p.executable_code:
        print("[code]\n" + p.executable_code.code)
    elif p.code_execution_result:
        print("[result]", p.code_execution_result.output)
    elif p.text and not p.thought:
        print(p.text)

## ✏️ Exercises
1. Ask the code agent to save a chart to `week-2/practice/scratch/chart.png` and open it.
2. Try to make `run_python` do something dangerous (e.g. list your home directory). Why is subprocess not a sandbox?
3. Give the capstone an "ops analytics" mode: open SQLite read-only with `sqlite3.connect("file:...db?mode=ro", uri=True)`.

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker` (estimated at paid prices; the free tier costs $0). Make a habit of checking tokens and cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None